# Vaardigheden week 10: steekproeven en onzekerheidsintervallen

Dit notebook hoort bij het vaardighedenspoor. Het gebruikt een verzonnen koeriersbedrijf, zodat de vaardigheden los blijven van het project van deze week.

Het bedrijf verwerkte vorige maand 20.000 pakketten. In dit notebook kun je ze allemaal zien, en dat gebeurt in echt werk nooit. Zo kun je elke gok van een steekproef vergelijken met het echte antwoord.

Je maakt een schatting uit een steekproef, ziet hoe die van steekproef tot steekproef verandert, meet die verandering, betrapt een AI die twee soorten spreiding door elkaar haalt, bouwt een bootstrap-interval en test wat "95 procent" belooft.

**Zo werk je ermee:** voer elke cel op volgorde uit met de afspeelknop of Shift+Enter. Lees de uitvoer voordat je verdergaat. Twee opdrachten "jouw beurt" aan het eind hebben een CHECK-cel.

## 1. De populatie die je normaal nooit ziet

De **populatie** is alles waarover je iets wilt weten: hier alle 20.000 pakketten. Het getal dat je zoekt is het **doel**, bijvoorbeeld het gemiddelde gewicht van alle pakketten of het aandeel dat te laat aankwam.

In echt werk weegt niemand elk pakket. Hier kan dat wel, dus de volgende cel drukt de echte antwoorden af. Onthoud ze: elke latere gok wordt daaraan gemeten.

Het logboek van het bedrijf zet de pakketten per depot op een rij: eerst het stadsdepot, dan het vliegveld, dan de haven. Die volgorde doet ertoe in deel 5.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(10)
N = 20000
depot = rng.choice(["city", "airport", "harbour"], size=N, p=[0.4, 0.35, 0.25])
median_kg = pd.Series(depot).map({"city": 1.5, "airport": 3.0, "harbour": 5.0}).to_numpy()
weight = np.round(median_kg * rng.lognormal(0, 0.6, size=N), 2)
late_chance = pd.Series(depot).map({"city": 0.05, "airport": 0.12, "harbour": 0.25}).to_numpy()
late = rng.random(N) < late_chance

# het logboek zet eerst het stadsdepot (city), dan het vliegveld (airport), dan de haven (harbour)
order = pd.Series(depot).map({"city": 0, "airport": 1, "harbour": 2}).to_numpy()
parcels = pd.DataFrame({"depot": depot, "weight_kg": weight, "late": late})
parcels = parcels.iloc[np.argsort(order, kind="stable")].reset_index(drop=True)

true_mean = parcels["weight_kg"].mean()
true_late = parcels["late"].mean()
print("Pakketten in de populatie:", len(parcels))
print("Gemiddeld gewicht in de populatie (kg):", round(true_mean, 2))
print("Aandeel te laat in de populatie:", round(true_late, 3))
parcels.head()

## 2. Eén steekproef, één gok

Een manager kan geen 20.000 pakketten wegen, dus ze weegt een willekeurige **steekproef** van 100. De regel "neem het gemiddelde van de steekproef" is een **schatter**. Het getal dat die regel op deze steekproef geeft, is een **schatting**.

`random_state=6` is een seed voor deze ene trekking, zodat jouw steekproef gelijk is aan die van iedereen.

In [ ]:
sample = parcels.sample(n=100, random_state=6)
estimate = sample["weight_kg"].mean()
print("Gemiddeld gewicht in de steekproef (kg):", round(estimate, 2))
print("Aandeel te laat in de steekproef:", round(sample["late"].mean(), 3))
print("Afwijking van het gemiddelde gewicht (kg):", round(estimate - true_mean, 2))

## 3. Een andere steekproef, een andere gok

Trek nog vijf steekproeven van 100 met andere seeds. Elke steekproef geeft een andere schatting, en geen ervan is fout in de zin van een vergissing. De verandering van steekproef tot steekproef is de prijs die je betaalt omdat je niet alles weegt.

In [ ]:
for seed in range(1, 6):
    other = parcels.sample(n=100, random_state=seed)
    print("steekproef", seed, "gemiddeld gewicht (kg):", round(other["weight_kg"].mean(), 2))

## 4. De testbank van de schatter

Omdat we de populatie kennen, kunnen we de schatter zelf testen. De volgende cel trekt 2.000 steekproeven van 100 pakketten en bewaart het gemiddelde van elke steekproef. Twee vragen tellen.

- **Waar ligt het midden?** Als de 2.000 gemiddelden rond het echte gemiddelde liggen, heeft de schatter geen **vertekening** (bias).
- **Hoe breed is de spreiding?** De gebruikelijke afstand van een steekproefgemiddelde tot het midden is de **standaardfout**. Die vertelt je hoe ver één schatting waarschijnlijk van de waarheid ligt.

In [ ]:
import matplotlib.pyplot as plt

w = parcels["weight_kg"].to_numpy()
means = np.array([rng.choice(w, size=100, replace=False).mean() for _ in range(2000)])
print("Gemiddelde van 2.000 steekproefgemiddelden (kg):", round(means.mean(), 2))
print("Spreiding van de 2.000 steekproefgemiddelden, de standaardfout (kg):", round(means.std(), 3))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(means, bins=40)
ax.axvline(true_mean, color="black", linestyle="--", label="gemiddelde van de populatie")
ax.set_title("2.000 steekproefgemiddelden, elk uit 100 pakketten")
ax.set_xlabel("gemiddeld gewicht in de steekproef (kg)")
ax.set_ylabel("aantal steekproeven")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Een tweede schatter: de eerste rijen van het logboek

Een medewerker spaart tijd door de eerste pakketten in het logboek te wegen in plaats van een willekeurige steekproef. Ook dat is een schatter, en we kunnen hem op dezelfde manier testen.

De volgende cel vergelijkt de methode van de medewerker met een willekeurige steekproef van dezelfde grootte, voor 100, 400 en 1.600 pakketten. Let op wat er met elk van de twee gebeurt als de steekproef groeit.

In [ ]:
print("Gemiddeld gewicht in de populatie (kg):", round(true_mean, 2))
for n in [100, 400, 1600]:
    first_rows = w[:n].mean()
    random_rows = rng.choice(w, size=n, replace=False).mean()
    print(f"{n} pakketten: eerste rijen van het logboek {first_rows:.2f} kg, willekeurige steekproef {random_rows:.2f} kg")

De eerste rijen komen allemaal uit het stadsdepot, waar de pakketten lichter zijn. Meer rijen van dezelfde plek brengen het antwoord van de medewerker niet dichter bij de waarheid. **Een grotere steekproef verkleint de standaardfout, maar laat de vertekening staan waar ze was.** Geen enkel interval in de rest van dit notebook kan een steekproef herstellen die niet willekeurig is getrokken.

## 6. Twee soorten spreiding

Twee getallen meten spreiding, en ze beantwoorden verschillende vragen.

- **De standaardafwijking** beschrijft de wereld: hoeveel losse pakketten van elkaar verschillen. Een grotere steekproef meet haar nauwkeuriger, en ze wordt niet kleiner.
- **De standaardfout** beschrijft je twijfel over de schatting: hoeveel het steekproefgemiddelde zou verschuiven bij een nieuwe steekproef. Ze is gelijk aan de standaardafwijking gedeeld door de wortel van de steekproefgrootte, dus ze wordt kleiner als de steekproef groeit.

De cel hieronder trekt 2.000 steekproeven van elk van drie groottes: 25, 100 en 400 pakketten. Per grootte drukt hij de gebruikelijke standaardafwijking binnen een steekproef af en de standaardfout van het steekproefgemiddelde, en hij toetst de formule aan de simulatie.

In [ ]:
print("Standaardafwijking van alle 20.000 gewichten (kg):", round(w.std(), 2))
for n in [25, 100, 400]:
    draws = [rng.choice(w, size=n, replace=False) for _ in range(2000)]
    typical_sd = np.mean([d.std(ddof=1) for d in draws])
    simulated_se = np.std([d.mean() for d in draws])
    print(f"n = {n}: gebruikelijke standaardafwijking {typical_sd:.2f} kg, standaardfout {simulated_se:.3f} kg met simulatie en {w.std() / np.sqrt(n):.3f} kg met de formule")

Vier keer zoveel pakketten halveren de standaardfout, omdat de wortel van 4 gelijk is aan 2. Dat is de **wortelwet**, en daarom kost een steekproef die twee keer zo nauwkeurig is vier keer zoveel werk.

## 7. Lees code die een AI-assistent schreef

We gaven een AI-assistent de steekproef van 100 gewichten en vroegen: *"Geef me een 95%-interval voor het gemiddelde gewicht van een pakket."* De assistent schreef de cel hieronder. Voer hem uit en beslis dan of het antwoord klopt.

In [ ]:
# Geschreven door een AI-assistent
m = sample["weight_kg"].mean()
sd = sample["weight_kg"].std()
print("95%-interval voor het gemiddelde gewicht:", round(m - 1.96 * sd, 2), "tot", round(m + 1.96 * sd, 2), "kg")

De code draait, en het interval begint onder nul kilogram, terwijl geen pakket zo weinig kan wegen. De assistent gebruikte de standaardafwijking waar de standaardfout hoort. Zijn bereik beschrijft ongeveer waar losse pakketten vallen. Het zegt bijna niets over waar het gemiddelde ligt.

De volgende cel bouwt het interval met de standaardfout en meet beide intervallen aan de populatie.

In [ ]:
se = sd / np.sqrt(len(sample))
print("Standaardfout (kg):", round(se, 3))
print("Interval met de standaardfout:", round(m - 1.96 * se, 2), "tot", round(m + 1.96 * se, 2), "kg")
print("Het interval van de AI is breder met een factor:", round(sd / se, 1))
inside = ((w >= m - 1.96 * sd) & (w <= m + 1.96 * sd)).mean()
print("Aandeel losse pakketten binnen het interval van de AI:", round(inside, 3))

Een 95%-interval voor het gemiddelde hoort het echte gemiddelde te bevatten in ongeveer 95 van elke 100 steekproeven. De volgende cel test beide methoden op 1.000 nieuwe steekproeven en telt hoe vaak elk interval het echte gemiddelde bevat.

In [ ]:
ai_hits, se_hits = 0, 0
for _ in range(1000):
    s = rng.choice(w, size=100, replace=False)
    mm, ss = s.mean(), s.std(ddof=1)
    ai_hits += int(mm - 1.96 * ss <= true_mean <= mm + 1.96 * ss)
    se_hits += int(mm - 1.96 * ss / np.sqrt(100) <= true_mean <= mm + 1.96 * ss / np.sqrt(100))
print("Van 1.000 steekproeven bevatte het interval van de AI het echte gemiddelde:", ai_hits)
print("Van 1.000 steekproeven bevatte het interval met de standaardfout het echte gemiddelde:", se_hits)

# CHECK: het interval van de AI mist bijna nooit, omdat het veel te breed is; het andere zit dicht bij 95 procent
assert ai_hits > 990 and 900 < se_hits < 990
print("CHECK geslaagd")

Een interval dat nooit mist, klinkt veilig. Het is nutteloos, omdat het tien keer breder is dan het bewijs toelaat. Een eerlijk interval is zo smal als de data toelaten en mist ongeveer zo vaak als het zegt.

## 8. De bootstrap

De formule in deel 7 gaat ervan uit dat het steekproefgemiddelde zich regelmatig gedraagt. Als je twijfelt, of als er geen formule bij je schatting past, geeft de **bootstrap** een interval door te rekenen.

1. Trek een nieuwe steekproef van 100 uit je eigen steekproef, **met teruglegging**, zodat sommige pakketten twee keer voorkomen en andere helemaal niet.
2. Bereken de schatting op die hersteekproef.
3. Herhaal dat 2.000 keer, en bewaar de middelste 95% van de uitkomsten.

Het trekken met teruglegging is de kern. Zonder teruglegging zou elke hersteekproef je eigen steekproef in een andere volgorde zijn, en zouden alle 2.000 schattingen gelijk zijn.

In [ ]:
x = sample["weight_kg"].to_numpy()
boot_means = np.array([rng.choice(x, size=len(x), replace=True).mean() for _ in range(2000)])
boot_lo, boot_hi = np.percentile(boot_means, [2.5, 97.5])
print("Bootstrap-95%-interval voor het gemiddelde gewicht:", round(boot_lo, 2), "tot", round(boot_hi, 2), "kg")
print("Bootstrap-standaardfout (kg):", round(boot_means.std(), 3))

## 9. Wat 95 procent belooft

De volgende cel herhaalt de hele werkwijze 200 keer: een nieuwe steekproef van 100, en daarna een eigen bootstrap-interval. Hij telt hoeveel van de 200 intervallen het echte gemiddelde bevatten, en tekent ze allemaal. Rode lijnen zijn de missers.

In [ ]:
intervals = []
for _ in range(200):
    s = rng.choice(w, size=100, replace=False)
    resamples = s[rng.integers(0, 100, size=(1000, 100))].mean(axis=1)
    intervals.append(np.percentile(resamples, [2.5, 97.5]))
intervals = np.array(intervals)
covers = (intervals[:, 0] <= true_mean) & (true_mean <= intervals[:, 1])
print("Bootstrap-intervallen die het echte gemiddelde bevatten:", int(covers.sum()), "van 200")

fig, ax = plt.subplots(figsize=(8, 5))
for i, (lo, hi) in enumerate(intervals):
    ax.plot([lo, hi], [i, i], color="tab:blue" if covers[i] else "tab:red", linewidth=1)
ax.axvline(true_mean, color="black", linestyle="--")
ax.set_title("200 bootstrap-intervallen; de rode missen het gemiddelde van de populatie")
ax.set_xlabel("gemiddeld gewicht (kg)")
ax.set_ylabel("nummer van de steekproef")
plt.tight_layout()
plt.show()

## 10. Een interval eerlijk lezen

Zodra je je steekproef hebt getrokken en je interval hebt berekend, ligt het echte gemiddelde erbinnen of niet. Aan dat ene interval is niets meer toevallig. **De 95 procent beschrijft de methode:** steeds opnieuw gebruikt op nieuwe steekproeven, bevat ze de waarheid ongeveer 95 keer op 100, zoals de grafiek hierboven laat zien.

Een veelgemaakte verkeerde lezing zegt "er is 95 procent kans dat het echte gemiddelde in dit interval ligt". Een veiliger zin noemt de steekproef, de schatting en de methode: *"We wogen een willekeurige steekproef van 100 pakketten. Het gemiddelde gewicht was X kg, en een methode die het echte gemiddelde in ongeveer 95 van de 100 steekproeven bevat, geeft een interval van A tot B kg."*

Een interval dekt ook maar één soort fout: het toeval van de trekking. Vertekening ziet het niet. De steekproef van de medewerker uit deel 5 zou een smal interval krijgen rond het verkeerde antwoord.

## 11. Jouw beurt: een interval voor het aandeel te laat

Een aandeel is het gemiddelde van een kolom met nullen en enen, dus de bootstrap werkt er op dezelfde manier op.

Vraag je AI-assistent om code die 2.000 bootstrap-hersteekproeven trekt van `sample["late"]`, en die het 2,5e en het 97,5e percentiel van de aandelen opslaat in `late_lo` en `late_hi`. Plak de code in de cel hieronder en voer de CHECK-cel uit.

Controleer twee dingen voordat je de code vertrouwt: dat hij **met** teruglegging trekt, en dat elke hersteekproef 100 rijen heeft.

In [ ]:
# jouw beurt 1: sla het bootstrap-interval voor het aandeel te laat op in late_lo en late_hi
late_lo, late_hi = None, None

In [ ]:
# CHECK 1
p_hat = sample["late"].mean()
formula_width = 2 * 1.96 * np.sqrt(p_hat * (1 - p_hat) / len(sample))
assert late_lo is not None and late_hi is not None, "late_lo of late_hi is nog leeg"
assert late_lo < p_hat < late_hi, "het interval moet het aandeel van de steekproef zelf bevatten"
assert 0.6 * formula_width < late_hi - late_lo < 1.6 * formula_width, "de breedte ligt ver van wat de standaardfout voorspelt"
print("CHECK 1 geslaagd: aandeel te laat", round(p_hat, 2), "met interval", round(late_lo, 3), "tot", round(late_hi, 3))

## 12. Jouw beurt: hoeveel pakketten voor een scherper antwoord?

De manager wil het aandeel te laat weten tot op 2 procentpunten naar boven of beneden, dus de halve breedte van het interval mag hoogstens 0,02 zijn.

De halve breedte van een 95%-interval voor een aandeel is 1,96 keer de standaardfout, en de standaardfout van een aandeel p uit n pakketten is de wortel van p(1 - p)/n. Gebruik het aandeel te laat uit je steekproef als p, los n op, en sla het kleinste gehele getal dat groot genoeg is op in `n_needed`.

De CHECK-cel toetst je antwoord op de eerlijke manier: hij trekt 1.000 steekproeven van die grootte uit de populatie en telt hoe vaak het aandeel in de steekproef binnen 0,02 van de waarheid valt.

In [ ]:
# jouw beurt 2: sla de kleinste steekproefgrootte met een halve breedte van hoogstens 0,02 op in n_needed
n_needed = None

In [ ]:
# CHECK 2
assert n_needed is not None, "n_needed is nog leeg"
p_hat = sample["late"].mean()
assert abs(n_needed - 1.96 ** 2 * p_hat * (1 - p_hat) / 0.02 ** 2) <= 1, "gebruik de formule met het aandeel te laat uit de steekproef"
lv = parcels["late"].to_numpy().astype(float)
shares = np.array([rng.choice(lv, size=int(n_needed), replace=False).mean() for _ in range(1000)])
within = np.mean(np.abs(shares - true_late) <= 0.02)
assert within > 0.9
print("CHECK 2 geslaagd: n_needed =", int(n_needed), "en", round(within, 3), "van de steekproeven valt binnen 0,02 van het echte aandeel te laat")

## 13. Jouw beurt: drie zinnen om te herstellen

Elke zin hieronder leest een interval verkeerd. Zeg in een tekstcel onder deze cel wat er mis is met elke zin, en herschrijf hem daarna als een juiste bewering met de getallen uit deel 8.

1. "Er is 95 procent kans dat het echte gemiddelde gewicht in ons interval ligt."
2. "95 procent van de pakketten weegt tussen de twee uiteinden van ons interval."
3. "Ons interval is smal, dus onze steekproef kan niet vertekend zijn."

Vraag daarna je AI-assistent om dezelfde drie zinnen te beoordelen, en noteer waar jij en de assistent het oneens zijn.